# 06 — Surrogate Explanations (Group-Level)

Fits interpretable surrogate models to mimic each black-box model's **predictions**
(not actuals) at Global and Decile levels. Surrogate coefficients / importances
serve as explanations of what the black-box learned.

### Surrogate routing
| model_type | group_vars present? | Surrogate |
|---|---|---|
| `Regression` | Yes | **RE Panel (MixedLM)** — random intercepts per group |
| `Regression` | No | **GLS / OLS** with HC3 robust errors |
| `Classification` | Yes / No | **Decision Tree** + **Logistic Regression (L1)** |

### Feature optimisation
Applied to all surrogate types. Dropped features appear with `coef=0`,
`importance=0`, `selected=False` — interpreted as *no significant impact*
on the black-box output.

| Surrogate | Method |
|---|---|
| RE / GLS | Drop features with p > 0.10; accept reduced model if BIC improves |
| Decision Tree | Keep top-N features covering 95% cumulative Gini importance |
| Logistic Regression | L1 penalty — zeroes insignificant features automatically |

**Output**: `Outputs/surrogate_importance.csv`  
**No inputs to change.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [1]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.shap_utils      import load_model_artifacts
from src.surrogate_utils import aggregate_surrogate

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

NOTEBOOK_DIR  = os.path.abspath('')
OUTPUT_DIR    = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_SAMPLES = 30   # minimum rows in a group to fit a surrogate

print('Libraries loaded')

Libraries loaded


## 1. Load Manifesto

In [2]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']

print(f'Runs scheduled : {len(dataset_configs)}')
print(f'Model types    : {sorted(set(r["model_type"] for r in dataset_configs))}')
print(f'Model families : {sorted(set(r["model_family"] for r in dataset_configs))}')

Runs scheduled : 21
Model types    : ['Regression']
Model families : ['Linear', 'Neural', 'Tree']


## 2. Surrogate Pipeline — All Runs

In [3]:
all_results     = []
artifacts_cache = {}   # model_name -> (model_obj, features, aux)

for run in dataset_configs:
    run_id     = run['run_id']
    model_name = run['model']
    model_type = run['model_type']
    group_var  = run['group_vars'][0] if run['group_vars'] else None

    print(f'\n[{run_id}]  type={model_type}  group_var={group_var}', flush=True)

    # ── Load prediction file (from manifest path — no hardcodes) ──────────────
    df = pd.read_csv(run['path'])

    # ── Load feature list from model pkl (cached per model) ───────────────────
    if model_name not in artifacts_cache:
        print(f'  Loading model artifacts...', flush=True)
        _, features, _ = load_model_artifacts(run)
        artifacts_cache[model_name] = features
        print(f'  Features : {len(features)}', flush=True)
    else:
        features = artifacts_cache[model_name]

    # ── Fit surrogates at Global + Decile levels ───────────────────────────────
    agg_df = aggregate_surrogate(df, features, run, min_samples=MIN_SAMPLES)

    if agg_df.empty:
        print(f'  [warn] no groups met min_samples={MIN_SAMPLES}', flush=True)
        continue

    all_results.append(agg_df)

    for gvar, grp in agg_df.groupby('Group_Variable'):
        n_buckets  = grp['Group_Value'].nunique()
        surr_types = grp['Surrogate_Model'].unique().tolist()
        avg_fid_r2  = grp['Fidelity_R2'].mean()
        avg_fid_acc = grp['Fidelity_Accuracy'].mean()
        fid_str = (f'Fidelity_R2={avg_fid_r2:.3f}' if not np.isnan(avg_fid_r2)
                   else f'Fidelity_Acc={avg_fid_acc:.3f}')
        print(f'  [{gvar}] {n_buckets} bucket(s)  surrogates={surr_types}  {fid_str}',
              flush=True)

print('\nSurrogate pipeline complete.')


[FE_LSDV_DEV]  type=Regression  group_var=Segment


  Loading model artifacts...


  Features : 38


  [Decile] 10 bucket(s)  surrogates=['RE_Panel', 'GLS']  Fidelity_R2=0.308


  [Global] 1 bucket(s)  surrogates=['RE_Panel']  Fidelity_R2=0.906


  [RE_Segment] 5 bucket(s)  surrogates=['RE_Panel']  Fidelity_R2=0.906


... (67 outputs left out)

  [Global] 1 bucket(s)  surrogates=['GLS']  Fidelity_R2=0.962



[LSTM_OOT_05]  type=Regression  group_var=Segment


  [Global] 1 bucket(s)  surrogates=['GLS']  Fidelity_R2=0.927



Surrogate pipeline complete.


## 3. Save Output

In [4]:
surr_df = pd.concat(all_results, ignore_index=True)

OUTPUT_COLS = [
    'Dataset', 'Model', 'Model_Domain', 'Model_Type',
    'Group_Variable', 'Group_Value', 'Samples',
    'Surrogate_Model', 'Feature', 'Selected',
    'Surrogate_Coef', 'Feature_Importance',
    'Fidelity_R2', 'Fidelity_Accuracy', 'BIC',
]
surr_df = surr_df[[c for c in OUTPUT_COLS if c in surr_df.columns]]
surr_df = surr_df.sort_values(
    ['Dataset', 'Model', 'Surrogate_Model', 'Group_Variable', 'Group_Value', 'Feature_Importance'],
    ascending=[True, True, True, True, True, False]
).reset_index(drop=True)

out_path = os.path.join(OUTPUT_DIR, 'surrogate_importance.csv')
surr_df.to_csv(out_path, index=False)

n_global = (surr_df['Group_Variable'] == 'Global').sum()
n_decile = (surr_df['Group_Variable'] == 'Decile').sum()
print(f'Saved: surrogate_importance.csv')
print(f'  Total rows   : {len(surr_df):,}')
print(f'  Global rows  : {n_global:,}')
print(f'  Decile rows  : {n_decile:,}')
print(f'  Surrogate types : {surr_df["Surrogate_Model"].unique().tolist()}')
print(f'  Selected features (any) : {surr_df["Selected"].sum():,}')

Saved: surrogate_importance.csv
  Total rows   : 8,969
  Global rows  : 749
  Decile rows  : 6,420
  Surrogate types : ['GLS', 'RE_Panel']
  Selected features (any) : 4,480


## 4. Quick Views

In [5]:
# Top selected features at Global level by model and surrogate type
global_sel = surr_df[
    (surr_df['Group_Variable'] == 'Global') & (surr_df['Selected'] == True)
]
print('── Top 10 Selected Features (Global, sorted by importance) ──────────')
display(
    global_sel.groupby(['Dataset', 'Model', 'Surrogate_Model'])
    .apply(lambda x: x.nlargest(10, 'Feature_Importance')
                      [['Feature', 'Surrogate_Coef', 'Feature_Importance',
                        'Fidelity_R2', 'Fidelity_Accuracy', 'BIC']])
    .reset_index(level=[0, 1, 2])
)

── Top 10 Selected Features (Global, sorted by importance) ──────────


,Dataset,Model,Surrogate_Model,Feature,Surrogate_Coef,Feature_Importance,Fidelity_R2,Fidelity_Accuracy,BIC
380,DEV,FE_LSDV,RE_Panel,NewYear,1.8603e+02,7.7402e+01,0.9063,NaN,298102.95
381,DEV,FE_LSDV,RE_Panel,is_year_start,-1.2376e+02,2.8526e+01,0.9063,NaN,298102.95
382,DEV,FE_LSDV,RE_Panel,any_holiday,-6.2598e+01,2.7771e+01,0.9063,NaN,298102.95
383,DEV,FE_LSDV,RE_Panel,Christmas,6.2174e+01,1.5325e+01,0.9063,NaN,298102.95
384,DEV,FE_LSDV,RE_Panel,September,5.7458e+01,1.3672e+01,0.9063,NaN,298102.95
...,...,...,...,...,...,...,...,...,...
8935,OOT_05,XGBoost,GLS,Weekly_Index,5.8031e+07,1.1269e+07,0.9688,NaN,1273.98
8936,OOT_05,XGBoost,GLS,Christmas,1.7291e+08,1.0961e+07,0.9688,NaN,1273.98
8937,OOT_05,XGBoost,GLS,VetreansDay,6.5910e+08,9.0828e+06,0.9688,NaN,1273.98
8938,OOT_05,XGBoost,GLS,Thanksgiving,2.1029e+08,3.2843e+06,0.9688,NaN,1273.98


In [6]:
# Fidelity summary — how well does each surrogate approximate the black-box?
fid_cols = ['Dataset', 'Model', 'Surrogate_Model', 'Group_Variable', 'Fidelity_R2', 'Fidelity_Accuracy', 'BIC']
fid_summary = (
    surr_df[fid_cols].drop_duplicates()
    .groupby(['Dataset', 'Model', 'Surrogate_Model', 'Group_Variable'])
    .first().reset_index()
)
print('── Surrogate Fidelity Summary ────────────────────────────────────────')
display(fid_summary)

── Surrogate Fidelity Summary ────────────────────────────────────────


,Dataset,Model,Surrogate_Model,Group_Variable,Fidelity_R2,Fidelity_Accuracy,BIC
0,DEV,FE_LSDV,GLS,Decile,0.9632,NaN,27621.59
1,DEV,FE_LSDV,RE_Panel,Decile,0.8079,NaN,-9685.09
2,DEV,FE_LSDV,RE_Panel,Global,0.9063,NaN,298102.95
3,DEV,FE_LSDV,RE_Panel,RE_Segment,0.9063,NaN,298102.95
4,DEV,LSTM,GLS,Decile,0.4861,NaN,27527.51
...,...,...,...,...,...,...,...
58,OOT_04,XGBoost,GLS,Decile,0.6041,NaN,359.59
59,OOT_04,XGBoost,GLS,Global,0.9906,NaN,4483.49
60,OOT_05,FE_LSDV,GLS,Global,0.9762,NaN,2175.02
61,OOT_05,LSTM,GLS,Global,0.9271,NaN,1612.76
